# 02 · Tools and the MCP server

**One function, two transports.** `tool_function(source)` wraps `source.search` so it returns JSON text
while keeping its typed signature. That single function becomes:

```
                         ┌──► LangChain StructuredTool   (tools.mode: direct, in-process)
source.search ─► tool ───┤
                         └──► MCP tool on FastMCP        (tools.mode: mcp, any MCP client can use it)
```

**What you will learn**
1. How a typed Python signature becomes the JSON schema the LLM sees
2. The MCP server's tools, resource and prompt, in-process
3. MCP over HTTP, the way the Docker setup runs it

In [1]:
# Setup: make the project importable and quiet the HTTP logs
import sys, logging, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
logging.getLogger("httpx").setLevel(logging.WARNING)

def show(obj, limit=1500):
    """Pretty-print a pydantic model, dict or list, cut to `limit` characters."""
    if hasattr(obj, "model_dump"):
        obj = obj.model_dump()
    if isinstance(obj, list):
        obj = [o.model_dump() if hasattr(o, "model_dump") else o for o in obj]
    text = json.dumps(obj, indent=2, ensure_ascii=False, default=str)
    print(text if len(text) <= limit else text[:limit] + "\n... (cut)")

In [2]:
from pharma_agent.config import load_config
from pharma_agent.wiring import build_http, build_store
from pharma_agent.sources import build_sources
from pharma_agent.tools import tool_function, langchain_tools, parse_evidence
import inspect

cfg = load_config(overrides={"store.path": ":memory:"})
sources = build_sources(cfg["sources"], build_http(cfg, build_store(cfg)))

## 1. The wrapped function keeps the typed signature

In [3]:
fn = tool_function(sources[0])
print(fn.__name__)
print(inspect.signature(fn))

search_drug_labels
(drug: typing.Annotated[str, FieldInfo(annotation=NoneType, required=True, description="Generic or brand name, e.g. 'metformin' or 'Eliquis'")], section: Annotated[Literal['indications', 'dosage', 'contraindications', 'warnings', 'adverse_reactions', 'drug_interactions', 'pregnancy', 'mechanism'], FieldInfo(annotation=NoneType, required=True, description='Which label section to return')] = 'drug_interactions', limit: typing.Annotated[int, FieldInfo(annotation=NoneType, required=True, description='How many labels to return', metadata=[Ge(ge=1), Le(le=3)])] = 1) -> str


## 2. As LangChain tools: the argument schema comes from that signature

In [4]:
tools = langchain_tools(sources)
for t in tools:
    print(t.name)
    for arg, spec in t.args.items():
        print(f"    {arg}: {spec.get('type', spec.get('enum', ''))}  - {spec.get('description', '')}")

search_drug_labels
    drug: string  - Generic or brand name, e.g. 'metformin' or 'Eliquis'
    section: string  - Which label section to return
    limit: integer  - How many labels to return
get_adverse_event_counts
    drug: string  - Generic drug name, e.g. 'atorvastatin'
    top_n: integer  - How many reactions to return
search_clinical_trials
    condition: string  - Disease or condition, e.g. 'type 2 diabetes'
    intervention: string  - Drug or other intervention, e.g. 'semaglutide'
    status: string  - Recruitment status filter
    limit: integer  - How many studies to return
search_pubmed
    query: string  - PubMed search terms, e.g. 'metformin lactic acidosis risk'
    limit: integer  - How many articles to return


In [5]:
text = tools[0].invoke({"drug": "warfarin", "section": "warnings"})
print(text[:400])
print("\nParsed back into Evidence:", [e.id for e in parse_evidence(text)])

{"results": [{"id": "fda-label:21255cd5:warnings", "source": "openfda_labels", "title": "FDA label: Warfarin Sodium (WARFARIN SODIUM), warnings", "text": "[BOXED_WARNING] WARNING: BLEEDING RISK Warfarin sodium can cause major or fatal bleeding [see Warnings and Precautions ( 5.1 )] . Perform regular monitoring of INR in all treated patients [see Dosage and Administration ( 2.1 )] . Drugs, dietary 

Parsed back into Evidence: ['fda-label:21255cd5:warnings']


## 3. The MCP server, in-process
`build_server()` registers the same functions on a FastMCP server, plus a **resource** (`pharma://sources`)
and a **prompt** (`drug_safety_review`). Here we call it directly, without a transport.

In [6]:
from pharma_agent.mcp_server import build_server
server = build_server(cfg)

for t in await server.list_tools():
    print(f"{t.name:26s} args: {list(t.inputSchema['properties'])}")

search_drug_labels         args: ['drug', 'section', 'limit']
get_adverse_event_counts   args: ['drug', 'top_n']
search_clinical_trials     args: ['condition', 'intervention', 'status', 'limit']
search_pubmed              args: ['query', 'limit']


In [7]:
resource = await server.read_resource("pharma://sources")
print(list(resource)[0].content[:400])

[
  {
    "source": "OpenFDALabels",
    "tool": "search_drug_labels",
    "description": "Read one section of the official FDA drug label (prescribing information) for a drug. Use for indications, dosage, contraindications, warnings (incl. boxed warning), adverse reactions, drug interactions, pregnancy, or mechanism of action."
  },
  {
    "source": "OpenFDAAdverseEvents",
    "tool": "get_adver


In [8]:
prompt = await server.get_prompt("drug_safety_review", {"drug": "adalimumab"})
print(prompt.messages[0].content.text)

Review the safety of adalimumab. 1) Read the boxed warning and warnings from its FDA label. 2) Get the most reported FAERS reactions. 3) List recruiting trials. Cite every source id.


## 4. MCP over HTTP (how Docker runs it)
Start the server as its own process, then connect with the same client library the agents use.

In [9]:
import subprocess, socket, time

server_proc = subprocess.Popen(
    [sys.executable, "-m", "pharma_agent.mcp_server", "--transport", "streamable-http", "--port", "8765"],
    cwd=ROOT, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
for _ in range(60):  # wait until the port accepts connections
    try:
        socket.create_connection(("127.0.0.1", 8765), timeout=0.5).close(); break
    except OSError:
        time.sleep(0.5)
print("MCP server running, pid", server_proc.pid)

MCP server running, pid 21348


In [10]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({"pharma": {"transport": "streamable_http", "url": "http://127.0.0.1:8765/mcp"}})
mcp_tools = await client.get_tools()
print("Tools over MCP:", [t.name for t in mcp_tools])

result = await next(t for t in mcp_tools if t.name == "search_clinical_trials").ainvoke(
    {"intervention": "tirzepatide", "status": "RECRUITING", "limit": 2})
print([e.id for e in parse_evidence(result)])

Received session ID: b670c499f84b4cd68052b08e575159c5


Negotiated protocol version: 2025-11-25


Received session ID: 6d20e11b27d943c28223a9f9e1d42e63


Negotiated protocol version: 2025-11-25


Tools over MCP: ['search_drug_labels', 'get_adverse_event_counts', 'search_clinical_trials', 'search_pubmed']
['nct:NCT06374875', 'nct:NCT05933499']


In [11]:
server_proc.terminate(); server_proc.wait()
print("MCP server stopped")

MCP server stopped


## Use these tools from Claude Desktop or Cursor
Add this to the client's MCP config (adjust the paths):
```json
{"mcpServers": {"pharma": {"command": "E:/pharma_agent/.venv/Scripts/python.exe",
                           "args": ["-m", "pharma_agent.mcp_server"], "cwd": "E:/pharma_agent"}}}
```
Next: **03_agents_and_graph.ipynb** gives these tools to LLM agents.